# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [5]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

tablas = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type = 'table'",
    conexion
)

print("tablas")
print(tablas)


conexion.close()





tablas
        name
0  pacientes
1    medicos
2     turnos


In [18]:
import sqlite3
import pandas as pd
 
conexion = sqlite3.connect("medicare.db")

estructura = pd.read_sql(
    "PRAGMA table_info(pacientes)",
    conexion
)

estructura2 = pd.read_sql(
    "PRAGMA table_info(medicos)",
    conexion
)

estructura3 = pd.read_sql(
    "PRAGMA table_info(turnos)",
    conexion
)

print(estructura)
print(estructura2)
print(estructura3)

conexion.close()


   cid              name     type  notnull dflt_value  pk
0    0                id  INTEGER        0       None   1
1    1            nombre     TEXT        0       None   0
2    2               dni     TEXT        0       None   0
3    3  fecha_nacimiento     TEXT        0       None   0
4    4            ciudad     TEXT        0       None   0
5    5       obra_social     TEXT        0       None   0
   cid          name     type  notnull dflt_value  pk
0    0            id  INTEGER        0       None   1
1    1        nombre     TEXT        0       None   0
2    2  especialidad     TEXT        0       None   0
3    3     matricula     TEXT        0       None   0
4    4        activo     TEXT        0       None   0
   cid         name     type  notnull dflt_value  pk
0    0           id  INTEGER        0       None   1
1    1  paciente_id  INTEGER        0       None   0
2    2    medico_id  INTEGER        0       None   0
3    3        fecha     TEXT        0       None   0
4    

In [6]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

for tabla in tablas['name']:
    filas = pd.read_sql(
        f"SELECT COUNT(*) AS total FROM '{tabla}'",
        conexion
    )

    print(f"{tabla}: {filas['total'][0]} filas")

conexion.close()

pacientes: 66 filas
medicos: 20 filas
turnos: 210 filas


**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.

Hallazgos:

1. Se encontraron 3 tablas en la base de datos: Pacientes, Medicos, Turnos.
2. Se encontraron datos de tipo numericos enteros y datos de tipo texto
3. Se encontraron 66 filas en la tabla "pacientes" - 20 filas en la tabla "medicos" - 210 filas en la tabla "turnos"

### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [81]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

df_pacientes = pd.read_sql("SELECT * FROM pacientes", conexion)
df_medicos = pd.read_sql("SELECT * FROM medicos", conexion)
df_turnos = pd.read_sql("SELECT * FROM turnos", conexion)

display(df_pacientes.head())
display(df_medicos.head())
display(df_turnos.head())


,id,nombre,dni,fecha_nacimiento,ciudad,obra_social
0,1,Martina García,28.341.992,1985-04-12,Buenos Aires,OSDE
1,2,Roberto Díaz,31.882.441,1990-11-30,Córdoba,Swiss Medical
2,3,Luciana Pérez,35.112.008,2001-07-22,NaN,PAMI
3,4,Martina García,28.341.992,1985-04-12,Buenos Aires,OSDE
4,5,Tomás Ruiz,40.223.117,2000-03-15,Rosario,NaN


,id,nombre,especialidad,matricula,activo
0,1,Dra. Beatriz Sosa,Clínica Médica,MP-12345,1
1,2,Dr. Nicolás Vera,Pediatría,MP-23456,1
2,3,Dra. Carmen Aguirre,Cardiología,NaN,1
3,4,Dr. Pablo Méndez,Clínica Médica,MP-45678,0
4,5,Dra. Beatriz Sosa,Clínica Médica,MP-12345,1


,id,paciente_id,medico_id,fecha,hora,motivo,costo,estado
0,1,1,1,2024-03-01,09:00,Control anual,3500,realizado
1,2,2,2,2024-03-01,10:30,Fiebre,4200,realizado
2,3,3,1,2024-03-02,08:30,Dolor abdominal,3500,realizado
3,4,4,3,2024-03-02,11:00,Electrocardiograma,8500,realizado
4,5,5,2,2024-03-03,NaN,Control pediátrico,4200,realizado


Vamos a ver cuantos nulos tenemos en cada tabla

In [94]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

print(f"\nDatos nulos en Pacientes:\n{df_pacientes.isnull().sum()}")
print(f"\nDatos nulos en Medicos:\n{df_medicos.isnull().sum()}")
print(f"\nDatos nulos en Turnos:\n{turnos.isnull().sum()}")



Datos nulos en Pacientes:
id                   0
nombre               0
dni                  7
fecha_nacimiento    11
ciudad               8
obra_social         12
dtype: int64

Datos nulos en Medicos:
id              0
nombre          0
especialidad    0
matricula       2
activo          4
dtype: int64

Datos nulos en Turnos:
id              0
paciente_id     0
medico_id       0
fecha           0
hora           16
motivo         16
costo           9
estado          0
dtype: int64


Ahora vamos a ver cuanto representan esos nulos en porcentajes

In [69]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")


print(f"\nPorcentaje de nulos de Pacientes:\n{(df_pacientes.isnull().sum()/len(df) * 100).round()}")
print(f"\nPorcentaje de nulos de Medicos:\n{(df_medicos.isnull().sum()/len(df) * 100).round()}")
print(f"\nPortentaje de nulos de Turnos:\n{(df_turnos.isnull().sum()/len(df) *100).round()}")



Porcentaje de nulos de Pacientes:
id                   0.0
nombre               0.0
dni                 11.0
fecha_nacimiento    17.0
ciudad              12.0
obra_social         18.0
dtype: float64

Porcentaje de nulos de Medicos:
id              0.0
nombre          0.0
especialidad    0.0
matricula       3.0
activo          6.0
dtype: float64

Portentaje de nulos de Turnos:
id              0.0
paciente_id     0.0
medico_id       0.0
fecha           0.0
hora           24.0
motivo         24.0
costo          14.0
estado          0.0
dtype: float64


Ahora vamos a ver si existen filas duplicadas y cuales son

In [142]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

print(f"Duplicados de pacientes:{df_pacientes.duplicated(keep=False).sum()}" )
print(f"Duplicados de medicos:{df_medicos.duplicated(keep=False).sum()}")
print(f"Duplicados de Turnos:{df_turnos.duplicated(keep=False).sum()}")

Duplicados de pacientes:0
Duplicados de medicos:0
Duplicados de Turnos:0


Arriba nos marca que no existen duplicados que sean exactamentes iguales

Vamos a seguir buscando por filas columnas especificas

In [151]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

df_pacientes[
    df_pacientes['dni'].notna() &
    df_pacientes.duplicated(subset=["dni"], keep=False)
].sort_values(by="dni")

,id,nombre,dni,fecha_nacimiento,ciudad,obra_social
16,17,Pablo Álvarez,22.075.745,1975-01-08,Bahía Blanca,Galeno
52,53,Pablo Álvarez,22.075.745,1975-01-08,Buenos Aires,PAMI
39,40,Florencia Fernández,22.430.558,1956-06-19,Salta,Sancor Salud
51,52,Florencia Fernández,22.430.558,1956-06-19,Buenos Aires,Galeno
0,1,Martina García,28.341.992,1985-04-12,Buenos Aires,OSDE
3,4,Martina García,28.341.992,1985-04-12,Buenos Aires,OSDE
1,2,Roberto Díaz,31.882.441,1990-11-30,Córdoba,Swiss Medical
10,11,Roberto Díaz,31.882.441,1990-11-30,Córdoba,Swiss Medical
25,26,Franco López,34.149.848,NaN,Rosario,Medife
45,46,Franco López,34.149.848,NaN,NaN,Swiss Medical


In [148]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

df_medicos[
    df_medicos["matricula"].notna() &
    df_medicos.duplicated(
        subset=["matricula"],
        keep=False
    )
].sort_values(by="matricula")

,id,nombre,especialidad,matricula,activo
0,1,Dra. Beatriz Sosa,Clínica Médica,MP-12345,1
4,5,Dra. Beatriz Sosa,Clínica Médica,MP-12345,1


In [147]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

df_turnos[
    df_turnos.duplicated(
        subset=["paciente_id", "medico_id", "fecha", "hora"],
        keep=False
    )
].sort_values(
    by=["paciente_id", "fecha", "hora"]
)

,id,paciente_id,medico_id,fecha,hora,motivo,costo,estado
0,1,1,1,2024-03-01,09:00,Control anual,3500,realizado
10,11,1,1,2024-03-01,09:00,Control anual,3500,realizado


Vemos si hay datos de tipo incorrecto

In [146]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

print(f"\nTipo de datos de pacientes:\n{df_pacientes.dtypes}")
print(f"\nTipo de datos de medicos:\n{df_medicos.dtypes}")
print(f"\nTipo de datos de :\n{df_turnos.dtypes}")





Tipo de datos de pacientes:
id                  int64
nombre                str
dni                   str
fecha_nacimiento      str
ciudad                str
obra_social           str
dtype: object

Tipo de datos de medicos:
id              int64
nombre            str
especialidad      str
matricula         str
activo            str
dtype: object

Tipo de datos de :
id             int64
paciente_id    int64
medico_id      int64
fecha            str
hora             str
motivo           str
costo            str
estado           str
dtype: object


Tenemos datos con tipos incorrectos que podriamos revisar

Tabla de pacientes: puede ser "fecha_nacimiento" que esta como un dato de tipo texto(str), no es que este mal pero si queremos operar con fechas no podriamos hacerlo, habria que convertirlo a un tipo de dato "datetime"

Tabla de medicos: Puede ser "activo" que se encuentra como un dato de tipo texto(str), podriamos revisarlo y si es un "True o False" convertirlo en un dato de tipo booleano

Tabla de turnos: En turnos el costo esta como un dato de tipo texto(str), debemos convertirlo a un dato de tipo float para poder operar con numeros. Tambien en caso de ser necesario y si debemos operar con esas columnas podemos revisar la fecha y hora que estan como un dato de tipo texto(str), en fecha al igual que la hora podemos pasarlo a un datetime

Vamos a revisar si aparecen errores aunque no sean nulos

In [ ]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect("medicare.db")

#df_turnos["costo"].describe()
df_turnos["costo"].unique()

<StringArray>
[ '3500',  '4200',  '8500',     nan,  '5500',  '-500',   'N/A',  '9200',
  '6800',      '', '-3500', '-1000']
Length: 12, dtype: str

Si, encontre 3 cosas

1. Los datos estan almacenados como datos de tipo texto(str)
2. Tenemos datos faltantes en "nan, N/A" y cadenas vacias
3. Encontramos valores sospechosos en los costos porque son negativos

**Antes de seguir al Paso 3, escriban una lista con todos los problemas que encontraron, uno por línea.** No empiecen a limpiar todavía.

Problemas encontrados en Pacientes:
*   La columna "fecha_nacimiento" esta almacenada como texto (str), no seria un problema, pero en caso de querer trabajar con la fecha seria conveniente poder convertirla a un tipo de dato datetime

Datos nulos en pacientes: 
 *      dni -7
 *      fecha_nacimiento - 11
 *      ciudad - 8
 *      obra_social - 12

 Encontramos 8 datos que estan duplicados teniendo mismo DNI y nombre

Problemas encontrados en Medicos:
*   La columna "activo" esta almacenada como un dato de tipo texto(str), seria conveniente convertirla en un dato de tipo booleano(bool)

Datos nulos encontrados en medicos:
*       matricula - 2
*       activo - 4

Encontramos 1 dato que tiene la misma matricula

Problemas encontrados en Turnos:
*   La columna fecha esta almacenada como un dato de tipo texto(str), en caso de necesitar trabajar con ella seria conveniente convertirla a un tipo de dato datetime.
*   La columna hora tambien esta almacenada como un tipo de dato texto(str), en caso de que tambien alla que trabar con ella lo mejor seria convertirla a un tipo de dato datetime
*   La columna costo esta almacenada como un dato de tipo texto(str) en lugar de un tipo numerico como float
*   La columna costo contiene valores faltantes representados como "nan, N/A" y cadenas vacias
*   La columna costo contiene valores negativos (-500, -3500, -1000) que son sospechosos porque representan costos de los turnos

Datos nulos encontrados en turnos:
*       hora - 16
*       motivo - 16
*       costo - 9

Encontramos 1 dato duplicado manteniendo la misma fecha/hora



### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| pacientes | ciudad | 8 nulos | Reemplazar por "desconocida" | Seria lo mejor para no inventar ciudades |
| pacientes | dni | 7 nulo | Mantener como nulos y revisarlo | No podemos inventar un DNI |
| pacientes | dni | DNI repetidos | Conservar un unico registro del DNI  | El DNI identifica a la persona y evita registrar al mismo paciente varias veces |
| pacientes | fecha_nacimiento | 11 | Mantener como nulos y revisarlo | No conocemos las fechas reales |
| pacientes | obra_social | 12 | Reemplazar por "Desconocida" o "No informada" | De esta manera diferenciamos la falta de informacion de una obra social |
| pacientes | dni | DNI repetidos | Conservar un unico registro del DNI  | El DNI identifica a la persona y evita registrar al mismo paciente varias veces |

*   

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| medicos | matricula | 2 nulos | Mantenemos como nulos y revisamos | No podemos inventar una matricula de un profesional |
| medicos | matricula | Matriculas repetidas | Conservar un unico registro por matricula | La matrícula identifica al profesional y permite detectar registros del mismo médico. |
| medicos | activo | 4 nulos | Mantenemos como nulos y revisamos | No sabemos si esos medicos estan activos o inactivos |
| medicos | activo | tipo texto(str) | Podriamos revisarlo y convertirlo en un dato booleano | Permite representar los 1 y 0 como verdadero/falso validando antes los valores |

*   

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| turnos | hora | 16 nulos | Mantenemos como nulos y revisamos | No conocemos el horario de los turnos |
| turnos | motivo | 16 nulos | Podriamos mantenerlo como nulos o marcarlo como "No informado" | No podemos invertar el motivo de la consulta |
| turnos | costo | 9 nulos | Mantenemos como nulos y revisar | No conocemos el importe real |
| turnos | costo | Tipo texto(str) | Convertir a un dato numerico | Necesitamos realizar calculos matematicos, si esta en un dato de tipo texto no podriamos hacerlo |
| turnos | costo | Valores N/A y vacios | convertir a nulos | Unificamos la representacion de los datos faltantes |
| turnos | costo | Valores negativos | Investigaria antes de corregirlo | No conocemos el importe correcto |
| turnos | fecha | Tipo texto(str) | Podriamos convertirlo a un tipo de dato datetime  | De esta manera nos permitiria ordenar y filtrar los turnos por fecha |
| turnos | hora | Tipo texto(str) | Podriamos convertirlo a un tipo de dato datetima | Facilitaria las operaciones con horarios |
| turnos | paciente_id/medico_id/fecha/hora | Dos registros del mismo turno | Eliminar uno de los registros duplicados| Evitar contabilizar dos veces un mismo turno. |


### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [154]:
# Limpieza de pacientes
df_pacientes_limpio = df_pacientes.copy()

# Limpieza de médicos
df_medicos_limpio = df_medicos.copy()
# Tu código acá

# Limpieza de turnos
df_turnos_limpio = df_turnos.copy()
# Tu código acá


> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [136]:
#tu codigo aqui

### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.

**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [ ]:
print("=" * 50)
print("REPORTE CLÍNICA MEDICARE")
print("=" * 50)

# tu reporte acá

→ [Ir a la solución](./solucion.md)